# Exploring how to scrape the "Ministère de l'Intérieur" website.

First, libraries, base url and path's initialisation

In [18]:
import subprocess
import sys

# Install selenium and webdriver-manager
subprocess.check_call([sys.executable, "-m", "pip", "install", "selenium", "webdriver-manager"])

0

In [ ]:
import os
import bs4
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import re
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

# Chemin vers le dossier contenant le notebook actuel
current_dir = os.getcwd()

starting_date = "2015-01-01"
ending_date = "2025-12-31"
main_url = f"https://www.interieur.gouv.fr/documentation?search=&sort_by=newest&field_publication_date%5Bmin%5D={starting_date}&field_publication_date%5Bmax%5D={ending_date}"

In [ ]:
# Use Selenium to bypass Cloudflare protection
print(f"Loading {main_url}...")

# Setup Chrome options
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')

# Create driver
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

try:
    driver.get(main_url)
    
    print("Waiting for cards to load...")
    # Wait for cards to be present (max 15 seconds)
    WebDriverWait(driver, 15).until(
        EC.presence_of_all_elements_located((By.CLASS_NAME, "fr-card"))
    )
    
    # Give it a bit more time to be safe
    time.sleep(2)
    
    # Get the page content
    html_content = driver.page_source
    
finally:
    driver.quit()

print(f"HTML length: {len(html_content)}")
print("\nSearching for 'fr-card'...")
print("'fr-card' found in HTML:", 'fr-card' in html_content)
print(f"\nFirst 1500 characters of HTML:")
print(html_content[:1500])

Loading https://www.interieur.gouv.fr/documentation?search=&sort_by=newest&field_publication_date%5Bmin%5D=2015-01-01&field_publication_date%5Bmax%5D=2025-12-31...
Waiting for cards to load...
HTML length: 236659

Searching for 'fr-card'...
'fr-card' found in HTML: True

First 1500 characters of HTML:
<html lang="fr" dir="ltr" prefix="og: https://ogp.me/ns#" data-fr-scheme="system" class="toolbar-loading toolbar-anti-flicker js" data-fr-js="true" data-fr-theme="light"><head>
    <meta charset="utf-8">
<noscript><style>form.antibot * :not(.antibot-message) { display: none !important; }</style>
</noscript><meta name="description" content="Le site officiel du ministère de l'Intérieur : actualités, votre sécurité, vos démarches administratives, collectivités territoriales, immigration, préfet, gendarmerie, police, sécurité civile, sapeurs-pompiers, secrétariat général, sécurité routière, élections.">
<meta name="abstract" content="Le site officiel du ministère de l'Intérieur : actualités, 

In [24]:
base_url = "https://www.interieur.gouv.fr"

# Parse HTML and extract card information
soup = bs4.BeautifulSoup(html_content, 'html.parser')

# Find all cards
cards = soup.find_all('div', class_='fr-card')

# Extract information from each card
cards_data = []
for card in cards:
    # Extract title and link
    title_elem = card.find('h3', class_='fr-card__title')
    link_elem = title_elem.find('a') if title_elem else None
    
    title = link_elem.get_text(strip=True) if link_elem else None
    link = link_elem.get('href') if link_elem else None
    link = base_url + link if link and link.startswith('/') else link
    # Extract date
    date_elem = card.find('em', class_='placeholder')
    date = date_elem.get_text(strip=True) if date_elem else None
    
    # Only add if we have at least title and link
    if title and link:
        cards_data.append({
            'titre': title,
            'date': date,
            'lien': link
        })

print(f"Total cards extracted: {len(cards_data)}")
print("\nFirst few cards:")
for card in cards_data[:3]:
    print(card)

# Save to JSON file
output_file = os.path.join(current_dir, 'ministere_interieur_cards.json')
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(cards_data, f, ensure_ascii=False, indent=2)

print(f"\nCards saved to: {output_file}")

Total cards extracted: 24

First few cards:
{'titre': 'BOMI Décembre 2025 n° 2 du 31 décembre 2025', 'date': '31 décembre 2025', 'lien': 'https://www.interieur.gouv.fr/documentation/bulletin-officiel-du-ministere-de-linterieur/bomi-decembre-2025-ndeg-2-du-31-decembre-2025.html'}
{'titre': 'Les haltes soins addictions', 'date': '23 décembre 2025', 'lien': 'https://www.interieur.gouv.fr/documentation/rapports/haltes-soins-addictions-dispositif-experimente-depuis-2016-pour-reduire-risques-et-nuisances.html'}
{'titre': 'Semaine du 15 au 21 décembre 2025', 'date': '16 décembre 2025', 'lien': 'https://www.interieur.gouv.fr/documentation/agenda/semaine-du-15-au-21-decembre-2025.html'}

Cards saved to: c:\Users\rorod\Documents\Pro\RDB Consulting\Mission Hélène 11-2025--02-2026\DIMIG (collab with Hélène)\Ministere_Interieur\ministere_interieur_cards.json
